In [1]:
import os
import itertools

import mlflow
import mlflow.sklearn
import pandas as pd
import sqlalchemy as sa
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("penguins-clasificacion")

engine = sa.create_engine(os.environ["DATA_DB_CONN"])
df = pd.read_sql("SELECT species, island, culmen_length_mm, culmen_depth_mm, "
                 "flipper_length_mm, body_mass_g, sex FROM clean.penguins_clean", engine)

print(f"Filas cargadas desde clean: {len(df)}")
print(f"MLflow: {mlflow.get_tracking_uri()}")
df.head()

2026/10/05 22:43:29 INFO mlflow.tracking.fluent: Experiment with name 'penguins-clasificacion' does not exist. Creating a new experiment.


Filas cargadas desde clean: 333
MLflow: http://mlflow:5000


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


In [2]:
NUMERICAS = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAS = ["island", "sex"]
OBJETIVO = "species"

X = df[NUMERICAS + CATEGORICAS]
y = df[OBJETIVO]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")
print(f"\nDistribucion en entrenamiento:\n{y_train.value_counts()}")

Entrenamiento: 266 | Prueba: 67

Distribucion en entrenamiento:
species
Adelie       117
Gentoo        95
Chinstrap     54
Name: count, dtype: int64


In [3]:
def construir_pipeline(n_estimators, max_depth, min_samples_split):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), NUMERICAS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
        ])),
        ("clf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1,
        )),
    ])


malla = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 5, 10],
}

combinaciones = list(itertools.product(*malla.values()))
print(f"Total de ejecuciones a realizar: {len(combinaciones)}\n")

resultados = []

for i, (n_est, prof, min_split) in enumerate(combinaciones, start=1):
    with mlflow.start_run(run_name=f"rf_{i:02d}_n{n_est}_d{prof}_s{min_split}"):
        modelo = construir_pipeline(n_est, prof, min_split)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)

        metricas = {
            "accuracy": accuracy_score(y_test, pred),
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, pred, average="macro"),
        }

        mlflow.log_params({
            "n_estimators": n_est,
            "max_depth": prof,
            "min_samples_split": min_split,
            "modelo": "RandomForestClassifier",
            "filas_train": len(X_train),
            "filas_test": len(X_test),
        })
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(modelo, artifact_path="modelo",
                                 input_example=X_train.head(3))

        resultados.append({**dict(zip(malla.keys(), (n_est, prof, min_split))), **metricas})
        print(f"[{i:02d}/{len(combinaciones)}] n={n_est} d={prof} s={min_split} "
              f"-> acc={metricas['accuracy']:.4f} f1={metricas['f1_macro']:.4f}")

print("\nExperimentacion completada")

Total de ejecuciones a realizar: 27



2026/10/05 22:45:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_01_n50_d3_s2 at: http://mlflow:5000/#/experiments/1/runs/b306c7c296fe425595a2ec73a1be9089.
2026/10/05 22:45:15 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[01/27] n=50 d=3 s=2 -> acc=1.0000 f1=1.0000


2026/10/05 22:45:18 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_02_n50_d3_s5 at: http://mlflow:5000/#/experiments/1/runs/fff335f0ad2c4b2e979d79b26472037a.
2026/10/05 22:45:18 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[02/27] n=50 d=3 s=5 -> acc=1.0000 f1=1.0000


...

[25/27] n=200 d=10 s=2 -> acc=1.0000 f1=1.0000


2026/10/05 22:46:38 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_26_n200_d10_s5 at: http://mlflow:5000/#/experiments/1/runs/d46894f787594afd82cb2083ea3893b5.
2026/10/05 22:46:38 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[26/27] n=200 d=10 s=5 -> acc=1.0000 f1=1.0000


2026/10/05 22:46:42 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_27_n200_d10_s10 at: http://mlflow:5000/#/experiments/1/runs/86444140b2a54c0ca05c36b2e61b3a31.
2026/10/05 22:46:42 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[27/27] n=200 d=10 s=10 -> acc=1.0000 f1=1.0000

Experimentacion completada


In [4]:
mlflow.set_experiment("penguins-dos-variables")

NUMERICAS_2 = ["culmen_depth_mm", "body_mass_g"]
CATEGORICAS_2 = []

X2 = df[NUMERICAS_2]
y2 = df[OBJETIVO]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

def pipeline_2v(n_estimators, max_depth, min_samples_split):
    return Pipeline([
        ("prep", ColumnTransformer([("num", StandardScaler(), NUMERICAS_2)])),
        ("clf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1,
        )),
    ])

resultados_2v = []

for i, (n_est, prof, min_split) in enumerate(combinaciones, start=1):
    with mlflow.start_run(run_name=f"rf2v_{i:02d}_n{n_est}_d{prof}_s{min_split}"):
        modelo = pipeline_2v(n_est, prof, min_split)
        modelo.fit(X2_train, y2_train)
        pred = modelo.predict(X2_test)

        metricas = {
            "accuracy": accuracy_score(y2_test, pred),
            "f1_macro": f1_score(y2_test, pred, average="macro"),
            "precision_macro": precision_score(y2_test, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y2_test, pred, average="macro"),
        }

        mlflow.log_params({
            "n_estimators": n_est,
            "max_depth": prof,
            "min_samples_split": min_split,
            "modelo": "RandomForestClassifier",
            "variables": ",".join(NUMERICAS_2),
            "n_variables": len(NUMERICAS_2),
        })
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(modelo, artifact_path="modelo",
                                 input_example=X2_train.head(3))

        resultados_2v.append({"n_estimators": n_est, "max_depth": prof,
                              "min_samples_split": min_split, **metricas})
        print(f"[{i:02d}/27] n={n_est} d={prof} s={min_split} "
              f"-> acc={metricas['accuracy']:.4f} f1={metricas['f1_macro']:.4f}")

print("\nSegunda experimentacion completada")

2026/10/05 22:49:05 INFO mlflow.tracking.fluent: Experiment with name 'penguins-dos-variables' does not exist. Creating a new experiment.


2026/10/05 22:49:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_01_n50_d3_s2 at: http://mlflow:5000/#/experiments/2/runs/ef1cc29f10984bf8ab2155dfd33a078d.
2026/10/05 22:49:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[01/27] n=50 d=3 s=2 -> acc=0.7910 f1=0.6019


2026/10/05 22:49:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_02_n50_d3_s5 at: http://mlflow:5000/#/experiments/2/runs/bcd7dd8efe6e4bc7b5ce18d449057456.
2026/10/05 22:49:12 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[02/27] n=50 d=3 s=5 -> acc=0.7910 f1=0.6019


...

[25/27] n=200 d=10 s=2 -> acc=0.7164 f1=0.6010


2026/10/05 22:50:27 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_26_n200_d10_s5 at: http://mlflow:5000/#/experiments/2/runs/f885660a62484f77a5edaa384bfe2a6f.
2026/10/05 22:50:27 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[26/27] n=200 d=10 s=5 -> acc=0.7463 f1=0.6172


2026/10/05 22:50:31 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_27_n200_d10_s10 at: http://mlflow:5000/#/experiments/2/runs/819380e2ab654422a0df515182b2162c.
2026/10/05 22:50:31 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[27/27] n=200 d=10 s=10 -> acc=0.7761 f1=0.6334

Segunda experimentacion completada
